# 13 — Ship it: a chat UI, and memory you control

**Scope** — who can reach your agent, and what it remembers between visits.

When you finish you can…

- turn any agent into a web chat app with one line, `agent.to_web()`, and check it without starting a server
- say what that built-in chat app does *not* do for you
- keep each chat's history on the server, in a SQLite file, with two small plain functions
- trim what the model sees with `ProcessHistory`, without losing anything you stored

> This notebook runs **keyless**: fake models stand in for a real one. One optional cell near the end
> uses your `MODEL`.

In [ ]:
from learningdantic import setup

MODEL = setup()

## One line: a web app

`agent.to_web()` returns a complete web app. It is a **Starlette** app (Starlette is the small web
framework that FastAPI is built on). It serves a chat page at `/` and a small JSON API under `/api/`.

You don't need a server to check that it works. Starlette's `TestClient` is a fake browser: it calls
the app directly in memory, with no port and no network. We only ask for `/api/...` addresses, because
`/` downloads the chat page from the internet.

The app also refuses any request addressed to a host name it doesn't know (status `421`), so a random
website can't reach the chat running on your laptop. `TestClient` calls itself `testserver` unless you
give it another address.

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.test import TestModel
from starlette.testclient import TestClient

app = Agent(TestModel()).to_web()  # the whole chat app, in one line

stranger = TestClient(app)  # addressed as "testserver"
print("as testserver:", stranger.get("/api/health").status_code)

browser = TestClient(app, base_url="http://localhost")  # addressed like your own browser
print("as localhost :", browser.get("/api/health").json())
print("configure    :", browser.get("/api/configure").json())

> ✅ **Checkpoint** — `as testserver: 421` (refused), then `as localhost : {'ok': True}`. The
> `configure` line lists one model, `test:test`: the chat page reads this to fill its model picker.

### Run it for real (in a terminal, not here)

A web server keeps running until you stop it, so it would freeze this notebook. Use a terminal.
Save this as `app.py` in the repo folder:

```python
from pydantic_ai import Agent

from learningdantic import setup

agent = Agent(setup(), instructions="Be brief and friendly.")
app = agent.to_web()
```

Then run `uv run uvicorn app:app` and open <http://127.0.0.1:8000>. **uvicorn** is the program that
listens on a port and hands each request to `app`. Press Ctrl+C to stop it. The docs call this UI
"meant for local development and debugging" ([web UI docs](https://pydantic.dev/docs/ai/web)).

## What the built-in chat app does *not* do

Know these two things before you show it to anyone:

- **The chat history lives in the browser.** Each time you send a message, the page sends the *whole*
  conversation along with it. The server stores nothing. Open another browser and the chat is gone.
- **Every user shares one `deps`.** `to_web(deps=...)` is fixed when the app is built, so the app
  can't give two users different data.

That is fine for you on your laptop. For real users, the server keeps the memory, filed under a
**chat id**. That is the rest of this notebook.

## Memory you control: `load` and `save`

Python ships with `sqlite3`: a whole database in one file, with nothing to install. One table, one row
per turn:

- `save(chat_id, result)` appends `result.new_messages_json()`: only this turn's messages, as JSON bytes.
- `load(chat_id)` reads that chat's rows in order and turns them back into messages with
  `ModelMessagesTypeAdapter` (you met it in 07).

If SQL is new to you: `CREATE TABLE` makes the table, `INSERT` adds a row,
`SELECT … WHERE chat_id = ? ORDER BY rowid` reads one chat's rows in the order they were added
(`rowid` is SQLite's built-in row number), and `commit()` writes the change to the file.

The `?` marks are placeholders: sqlite3 fills in the values safely. Never paste user text into SQL
yourself.

In [ ]:
import sqlite3
from pathlib import Path

from pydantic_ai.messages import ModelMessagesTypeAdapter

Path("../data").mkdir(exist_ok=True)
DB = Path("../data/13_chats.sqlite")
DB.unlink(missing_ok=True)  # start fresh each time this cell runs

db = sqlite3.connect(DB)
db.execute("CREATE TABLE turns (chat_id TEXT, messages BLOB)")  # BLOB = raw bytes
db.close()


# Each function opens the file, uses it and closes it again, like one web request would.
def save(chat_id, result):
    """Append this run's new messages to one chat."""
    db = sqlite3.connect(DB)
    db.execute("INSERT INTO turns VALUES (?, ?)", (chat_id, result.new_messages_json()))
    db.commit()
    db.close()


def load(chat_id):
    """Rebuild one chat's whole history, oldest turn first."""
    db = sqlite3.connect(DB)
    rows = db.execute(
        "SELECT messages FROM turns WHERE chat_id = ? ORDER BY rowid", (chat_id,)
    ).fetchall()
    db.close()
    history = []
    for row in rows:
        history += ModelMessagesTypeAdapter.validate_json(row[0])
    return history


print("chat 'ada' starts with", len(load("ada")), "messages")

> ✅ **Checkpoint** — `chat 'ada' starts with 0 messages`. A chat id nobody has used yet is just
> an empty history, so new chats need no special case.

## One turn of a chat

The one async rule again: `agent.run(...)` sits inside a function you wrote, so that function is
`async def`, and you `await` it when you call it. `load` and `save` stay plain `def`.

The fake model below (a `FunctionModel`, from 08) replies with every user prompt it was shown, so you
can *see* its memory.

In [ ]:
from pydantic_ai.messages import ModelResponse, TextPart, UserPromptPart
from pydantic_ai.models.function import FunctionModel


def echo_memory(messages, info):
    """A fake model: it answers with every user prompt it was shown."""
    prompts = []
    for message in messages:
        for part in message.parts:
            if isinstance(part, UserPromptPart):
                prompts.append(part.content)
    return ModelResponse(parts=[TextPart(f"I can see: {prompts}")])


agent = Agent(FunctionModel(echo_memory))


async def chat(agent, chat_id, prompt):
    """One turn: load the chat, run the agent on it, save the new messages."""
    result = await agent.run(prompt, message_history=load(chat_id))
    save(chat_id, result)
    return result.output


print(await chat(agent, "ada", "Hi, I'm Ada."))
print(await chat(agent, "ada", "What's my name?"))
print("stored for ada:", len(load("ada")), "messages")

> ✅ **Checkpoint** — the second reply shows both prompts, and ada has 4 stored messages (2 turns,
> each a request and a reply). No Python variable carried the chat between the two calls: the memory
> came from the file. Run the cell again and ada's chat simply carries on.

> **In a `.py` script**, Python can't `await` at the top level. There, call `agent.run_sync(...)`
> instead. Only there: `run_sync` fails inside a notebook, which is why this course never uses it.

## Trim what the model sees: `ProcessHistory`

Long chats get slow and expensive, because every turn sends the whole history again. `ProcessHistory`
is a capability (you met capabilities in 10). It runs a plain function on the messages before each
model request. Here the function keeps the newest 3: the last question, its answer, and the new prompt.

One catch: it doesn't only change what the model sees. It also **rewrites `result.all_messages()`**.
Watch:

In [ ]:
from pydantic_ai.capabilities import ProcessHistory


def keep_last_3(messages):
    """Keep the last question, its answer, and the new prompt."""
    return messages[-3:]


trimmed = Agent(FunctionModel(echo_memory), capabilities=[ProcessHistory(keep_last_3)])

history = load("ada")
result = await trimmed.run("Do you still know my name?", message_history=history)
print(result.output)
print("messages sent in:", len(history) + 1, "(the history + the new prompt)")
print("all_messages()  :", len(result.all_messages()), "(trimmed to 3, + the reply)")
print("new_messages()  :", len(result.new_messages()), "(this turn only)")

> ✅ **Checkpoint** — the model saw only `["What's my name?", 'Do you still know my name?']`.
> "Hi, I'm Ada." was trimmed away, so a real model would have forgotten the name: trimming saves
> tokens, and costs memory. `all_messages()` holds 4 messages, not 6. If `save` stored
> `all_messages()` in place of the chat, the introduction would be deleted for good. It stores
> `new_messages()` (2) and appends, so the file keeps everything.

## Trim the view, keep the file

Now use the trimmed agent inside `chat()`. The model's view stays small while the file keeps growing.

In [ ]:
for prompt in ["Tell me a joke.", "Another one!", "Last one, I promise."]:
    print(await chat(trimmed, "ada", prompt))
print("stored for ada:", len(load("ada")), "messages")

> ✅ **Checkpoint** — every reply shows just 2 prompts, yet ada has 10 stored messages: the whole
> chat. What the model sees and what you keep are now two separate choices, and both are yours.

## Try it with your real model (optional)

Same `chat()`, same file, your `MODEL`. Each call is a fresh `agent.run`, so the only way the model can
know your name in the second call is the database.

In [ ]:
import os

LIVE = not MODEL.startswith("openai:") or bool(os.getenv("OPENAI_API_KEY"))
if LIVE:
    real = Agent(MODEL, instructions="Be brief.")
    print(await chat(real, "sam", "Hi! I'm Sam and I love green tea."))
    print(await chat(real, "sam", "What's my name, and what do I like to drink?"))
    print("stored for sam:", len(load("sam")), "messages")
else:
    print("No key found — skipping the live part (the rest of this notebook ran keyless).")

> ✅ **Checkpoint** — with a key: a short greeting, then an answer like *"Your name is Sam, and you
> like green tea."*, and `stored for sam: 4 messages` (more if you run the cell again: the chat
> carries on). Your model's wording will differ. Without a key:
> the skip line.

## When you outgrow this

You now have the whole shape of a chat backend: a chat id, `load`, `await agent.run(...)`, `save`.
Swap the parts when you need to:

- **FastAPI** for many endpoints, request checking and docs pages. It is built on Starlette, so the app
  from `to_web()` and everything here carries over.
- **SQLModel or SQLAlchemy, with Postgres** for real tables and many server processes. `load` and
  `save` keep the same shape.
- **Your own chat page, with `deps` per user:** `VercelAIAdapter.dispatch_request(request, agent=agent,
  deps=...)` speaks the same protocol as the built-in page
  ([docs](https://pydantic.dev/docs/ai/integrations/ui/vercel-ai)).
- **Other ways to shape history:** the
  [message history docs](https://pydantic.dev/docs/ai/core-concepts/message-history).

## 🏋️ Exercise — two chats, one database

Use the trimmed agent and `chat()`. Start from an empty file: re-run the load/save cell first, or
reset the database at the top of your cell, so the counts are exact.

1. **No leaks.** Run two chats, `"alice"` and `"bob"`, 3 turns each, taking turns (alice, bob, alice,
   …). Put the chat's name in each prompt. `assert` that none of bob's replies mention alice, and the
   other way round.
2. **Trimming leaves the file alone.** `assert` that each chat still has all 6 messages stored
   (3 turns × 2), even though the model only ever saw the last 3.

*Stretch:* write a plain `def forget(chat_id)` that deletes one chat's rows. `assert` that `load` then
returns `[]` for that chat, while the other chat is untouched.

Solution: [`../solutions/13_two_chats.ipynb`](../solutions/13_two_chats.ipynb)

## Next →

→ **[14_capstone_support_desk.ipynb](14_capstone_support_desk.ipynb)** — put it all together: design a
support desk where every rule lives in the right place.